# Swin-T Nowcasting v2 — Colab

未知locationへの汎化を目的にした実験Notebookです。

- location名は入力に使わない
- location非重複かつ衛星構成を揃えた5-fold CV
- AHI / ABI / FCIで近い物理波長の6チャンネルを対応付け
- 衛星別stem・衛星embedding・fold内正規化
- UTCの月・時刻周期特徴と観測欠損フラグ
- 5-fold平均とSolafune提出形式の厳密検証

実装本体: `src/swin_nowcast_v2.py`

## 1. Colabセットアップ

ランタイムをGPUへ変更してから実行します。Hugging Faceトークンは不要です。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm rasterio scikit-learn
!nvidia-smi

## 2. パス設定

`ROOT`は`train_dataset/`, `evaluation_dataset/`, `sample_submission/`, `src/`を含むフォルダに変更してください。

In [ ]:
from pathlib import Path
import sys

ROOT = Path('/content/drive/MyDrive/solafune-workspace')
assert (ROOT / 'train_dataset').is_dir(), ROOT
assert (ROOT / 'evaluation_dataset').is_dir(), ROOT
assert (ROOT / 'sample_submission').is_dir(), ROOT
sys.path.insert(0, str(ROOT / 'src'))

from swin_nowcast_v2 import (
    Config, SATELLITE_BANDS, create_submission, ensemble_predict,
    get_device, make_folds, plot_histories, prepare_metadata, train_fold,
)

device = get_device()
print('Device:', device)
print('Band mapping:', SATELLITE_BANDS)

## 3. 設定とメタデータ

In [ ]:
config = Config(
    root=str(ROOT),
    batch_size=8,       # T4でOOMなら4
    epochs=30,
    workers=2,
    pretrained=True,
    use_amp=True,
)

train_df = prepare_metadata(config.train_dir / 'train_dataset.csv')
eval_df = prepare_metadata(config.evaluation_dir / 'evaluation_target.csv')
folds = make_folds(train_df, config.n_folds)

print('Train:', len(train_df), 'Evaluation:', len(eval_df))
print('Empty observations:', train_df.missing_observation.sum(), eval_df.missing_observation.sum())
for fold in folds:
    val = train_df.iloc[fold['validation_indices']]
    print(f"fold={fold['fold']} rows={len(val):,} locations={fold['validation_locations']} "
          f"satellites={val.satellite_target.value_counts().to_dict()}")

## 4. 軽量バンド・衛星補正アブレーション（任意）

本学習前に、同一fold・同一サンプルで次の3条件を比較します。

1. 旧Band 1–3＋共通正規化
2. 物理波長を対応付けた6バンド＋共通正規化
3. 対応6バンド＋衛星別正規化・Stem・衛星embedding

これは方向確認用の軽量検証です。最終判断は複数epochまたは全foldで行います。

In [ ]:
import os
import subprocess
import pandas as pd

ablation_output = ROOT / 'outputs/band_ablation/results.csv'
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src')
subprocess.run([
    sys.executable, str(ROOT / 'src/run_band_ablation.py'),
    '--root', str(ROOT),
    '--fold', '1',
    '--train-rows', '600',
    '--validation-rows', '300',
    '--epochs', '1',
    '--batch-size', '8',
    '--output', str(ablation_output.relative_to(ROOT)),
], check=True, env=env)

display(pd.read_csv(ablation_output).sort_values('best_validation_rmse'))

## 5. 学習

最初は`TRAIN_FOLDS = [0]`, `epochs=1`で動作確認し、その後5-foldへ変更します。正規化統計は各foldの学習locationだけから計算・保存されます。

In [ ]:
TRAIN_FOLDS = list(range(config.n_folds))

results = []
for fold_index in TRAIN_FOLDS:
    result = train_fold(config, train_df, folds[fold_index], device=device)
    results.append(result)

print({result['fold']: result['validation_rmse'] for result in results})
print('Mean CV:', sum(r['validation_rmse'] for r in results) / len(results))
plot_histories(results, config.model_dir / 'training_curves.png')

## 6. 5-fold推論

In [ ]:
INFERENCE_FOLDS = list(range(config.n_folds))
predictions = ensemble_predict(config, eval_df, INFERENCE_FOLDS, device=device)
print(predictions.shape, predictions.min(), predictions.mean(), predictions.max())

## 7. 提出ZIP作成

ZIP内部は必ず次の構造になります。

```text
submission_swin_v2.zip
├── evaluation_target.csv
└── test_files/
    └── CSVのgpm_imerg_filenameと完全一致するGeoTIFF群
```

`evaluation_target.csv`は配布ファイルを全列・全行・同じ順序でコピーします。GeoTIFFはsample submissionのprofileを維持します。

In [ ]:
submission_zip = ROOT / 'submission_swin_v2.zip'
create_submission(config, eval_df, predictions, submission_zip)
print('Created:', submission_zip)